In [1]:
# ==== 1. Lay code tu GitHub (env IP102_CODE_REPO) hoac /kaggle/input ====
import os, sys, glob, subprocess

CODE_DIR = None
os.environ['IP102_CODE_REPO'] = "https://github.com/nta2112/L2P-IP102-custom"

target = '/kaggle/working/L2P-for-IP102'
if os.environ.get('IP102_CODE_REPO'):
    repo = os.environ['IP102_CODE_REPO']
    if os.path.isdir(target):
        # Da clone roi -> PULL code moi nhat
        print('Pulling latest code...')
        subprocess.run(['git', '-C', target, 'pull'], check=True)
    else:
        # Chua clone -> clone moi
        repo = os.environ['IP102_CODE_REPO']
        print('git clone', repo)
        subprocess.run(['git', 'clone', repo, target], check=True)
    CODE_DIR = target
else:
    for base in ('/kaggle/input', '/kaggle/working'):
        found = sorted(glob.glob(os.path.join(base, '**', 'main_ip102.py'), recursive=True))
        if found:
            CODE_DIR = os.path.dirname(found[0])
            break
    if not CODE_DIR:
        raise RuntimeError('Khong tim thay code. Dat env IP102_CODE_REPO hoac day code vao /kaggle/input.')

sys.path.insert(0, CODE_DIR)
print('CODE_DIR =', CODE_DIR)

git clone https://github.com/nta2112/L2P-IP102-custom


Cloning into '/kaggle/working/L2P-for-IP102'...


CODE_DIR = /kaggle/working/L2P-for-IP102


In [2]:
# ==== 2. Cai dat thu vien JAX/Flax ====
import subprocess, sys
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

def need(mod):
    try:
        __import__(mod)
        return False
    except ImportError:
        return True

missing = [m for m in ('flax', 'jax', 'clu', 'ml_collections', 'tensorflow',
                       'scipy') if need(m)]
if missing:
    print('installing', missing)
    if 'jax' in missing and sys.platform.startswith('linux'):
        # Kaggle GPU notebooks: install JAX with CUDA 12 support
        try:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                            'jax[cuda12]', 'flax', 'clu', 'ml_collections',
                            'scipy'], check=True)
            missing = [m for m in ('flax', 'clu', 'ml_collections', 'scipy')
                       if need(m)]
        except subprocess.CalledProcessError:
            pass
    if missing:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] +
                       missing, check=True)
else:
    print('all deps present')

import jax
print('jax', jax.__version__, '| devices', jax.devices())

installing ['clu']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 6.1 MB/s eta 0:00:00
jax 0.7.2 | devices [CudaDevice(id=0)]


In [3]:
# ==== 3. Kiem tra dataset (auto tim bang deep-walk) ====
from libml.ip102_data import get_data_manager

dm = get_data_manager('ip102', seed=1993, split_val=True)
report = dm.verify()
print('DATA_ROOT =', dm.data_root)
for s in ('train', 'val', 'test'):
    if s not in report:
        continue
    r = report[s]
    print('%-5s images=%-5d anns=%-5d labeled=%-5d missing=%d'
          % (s, r['images_in_json'], r['annotations'],
             r['images_labeled'], r['missing_images']))
assert report['train']['missing_images'] == 0
assert report['num_classes'] == 25
print('num_classes =', report['num_classes'], '| val_split =', report['val_split'])

DATA_ROOT = /kaggle/input/datasets/nta212/ip102-for-object-detection
train images=8664  anns=9545  labeled=8664  missing=0
val   images=2176  anns=2374  labeled=2176  missing=0
test  images=2713  anns=2946  labeled=2713  missing=0
num_classes = 25 | val_split = val


In [4]:
# ==== 4. Test nhanh: max_tasks=1 (1 task dau), 1 epoch ====
from main_ip102 import run_train

# quick = run_train(model='L2P', max_tasks=1, memory_size=0, num_epochs=1)
# print('quick results ->', quick)

In [5]:
# HOTFIX: Patch ip102_eval.py để fix KeyError ngay lập tức
# HOTFIX: Patch ip102_eval.py để fix KeyError ngay lập tức
import sys

eval_path = '/kaggle/working/L2P-for-IP102/libml/ip102_eval.py'

# Đọc file gốc
with open(eval_path, 'r') as f:
    content = f.read()

# Patch 1: Thêm init Recall@1_unseen_energy vào evaluate_task
if "res['Recall@1_unseen_energy'] = None" not in content:
    content = content.replace(
        "res['Recall@1_seen'] = rec_seen",
        "res['Recall@1_seen'] = rec_seen\n    res['Recall@1_unseen_energy'] = None  # Hotfix KeyError"
    )
    print('✅ Patched: Added Recall@1_unseen_energy init')

# Patch 2: Đảm bảo write_results không lỗi khi key missing
if "row.get(k, None)" not in content:
    content = content.replace(
        "writer.writerow({k: _fmt(row[k]) for k in RESULTS_HEADER})",
        "writer.writerow({k: _fmt(row.get(k, None)) for k in RESULTS_HEADER})"
    )
    print('✅ Patched: write_results dùng row.get()')

# Ghi đè lại các thay đổi vào file
with open(eval_path, 'w') as f:
    f.write(content)

# Verify patch
with open(eval_path, 'r') as f:
    verify_content = f.read()

print('Recall@1_unseen_energy init:', "res['Recall@1_unseen_energy'] = None" in verify_content)
print('row.get() in write_results:', 'row.get(' in verify_content)
print('✅ HOTFIX DONE - Chạy cell 5 trở đi!')

✅ Patched: write_results dùng row.get()
Recall@1_unseen_energy init: True
row.get() in write_results: True
✅ HOTFIX DONE - Chạy cell 5 trở đi!


In [6]:
# ==== 5. Chay du: max_tasks=0 (toan bo task) ====
# Đảm bảo model đã được định nghĩa trước khi dùng                                                        
# ==== 5. Chay du: max_tasks=0 (toan bo task) ====
# Đảm bảo model đã được định nghĩa trước khi dùng                                                        
if 'model' not in locals() and 'model' not in globals():                                                 
    from main_ip102 import run_train  # import để model có sẵn qua side effect
if 'model' in locals() or 'model' in globals():
    model.mAP_matrix = []
    print("Đã reset mAP_matrix")
else:
    print("⚠️ Chưa có biến model, đang chờ chạy cell tạo model...")
full = run_train(model='L2P', max_tasks=0, memory_size=0)
print('final results ->', full)

⚠️ Chưa có biến model, đang chờ chạy cell tạo model...
✅ Đã khởi tạo model.mAP_matrix = []


KeyError: 'Recall@1_unseen_energy'

## Ket qua (results.csv)
Header: `task,numclass,cnn_top1,nme_top1,R@1,R@5,R@10,mAP,AUROC,FPR95,Plasticity,Forgetting,Overall`
(`AUROC/FPR95 = NA` khi da thay du toan bo lop -> khong con OOD de do).

In [ ]:
# ==== 6. Hien thi results.csv (glob dung duong dan noi code chay) ====
import glob
import pandas as pd

cands = set()
for base in ('/kaggle/working', CODE_DIR, '.'):
    cands |= set(glob.glob(os.path.join(base, '**', 'results.csv'),
                           recursive=True))
cands = sorted(cands, key=lambda p: os.path.getmtime(p))
print('results.csv files:', cands)
assert cands, 'Khong tim thay results.csv'
path = cands[-1]
print('displaying ->', path)
df = pd.read_csv(path)
display(df)